# Team's Names :

- Raghad Mohamed 
- Abdelrahman Wael 
- Hadeer Abdelhady
- Haneen Nasser  
- Ola Abdallah 


### Load the IMDb Movie Reviews Dataset
The IMDb dataset available through Keras is already preprocessed. Each review is represented as a sequence of integers, where each integer corresponds to a word in a dictionary. Words are ordered by overall frequency, so lower integers mean more frequent words.

`num_words=10000` means we'll only consider the top 10,000 most frequent words in the dataset. Rare words will be discarded.

In [ ]:
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences


(train_data, train_labels), (test_data, test_labels) = imdb.load_data(num_words=10000)

print(f"Training entries: {len(train_data)}, labels: {len(train_labels)}")
print(f"Test entries: {len(test_data)}, labels: {len(test_labels)}")

print("\nFirst training review (as integers):\n", train_data[0][:20], "...")
print("First training review label (0 = negative, 1 = positive):", train_labels[0])

print("\nLength of the first 5 training reviews:")
for i in range(5):
    print(f"Review {i}: {len(train_data[i])} words")

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Training entries: 25000, labels: 25000
Test entries: 25000, labels: 25000

First training review (as integers):
 [1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25] ...
First training review label (0 = negative, 1 = positive): 1

Length of the first 5 training reviews:
Review 0: 218 words
Review 1: 189 words
Review 2: 141 words
Review 3: 550 words
Review 4: 147 words


### Preprocessing: Decoding and Understanding the Data
The integers in the reviews represent words. Let's create a helper function to decode these integer sequences back into readable text. This helps confirm the tokenization and provides insight into the raw text before padding.

In [ ]:
# A dictionary mapping words to an integer index
word_index = imdb.get_word_index()

# The first indices are reserved
word_index = {k:(v+3) for k,v in word_index.items()}
word_index["<PAD>"] = 0
word_index["<START>"] = 1
word_index["<UNK>"] = 2  # unknown
word_index["<UNUSED>"] = 3

reverse_word_index = dict([(value, key) for (key, value) in word_index.items()])

def decode_review(text):
    return ' '.join([reverse_word_index.get(i, '?') for i in text])

# Let's decode and print the first training review
print("Decoded first training review:\n", decode_review(train_data[0]))

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 1s 1us/step
Decoded first training review:
 <START> this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert <UNK> is an amazing actor and now the same being director <UNK> father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for <UNK> and would recommend it to everyone to watch and the fly fishing was amazing really cried at the end it was so sad and you know what they say if you cry at a film it must have been good and this definitely was also <UNK> to the two little boy's that played the <UNK> of norman and paul they were just brilliant children are often left out of the <UNK> list i think because the stars that play them all grown up are such a big profil

### Preprocessing: Padding Sequences
Neural networks expect inputs to be of a uniform length. We need to pad the sequences so that all reviews have the same length. Reviews shorter than the `maxlen` will be padded with zeros, and reviews longer than `maxlen` will be truncated. I'll choose a `maxlen` of 256 words, which is a common length for text classification tasks.

In [ ]:
maxlen = 256

train_data = pad_sequences(train_data, value=word_index["<PAD>"], padding='post', maxlen=maxlen)
test_data = pad_sequences(test_data, value=word_index["<PAD>"], padding='post', maxlen=maxlen)

print(f"\nLength of the first training review after padding: {len(train_data[0])}")
print(f"Length of the second training review after padding: {len(train_data[1])}")

print("\nFirst training review after padding (first 20 elements):\n", train_data[0][:20], "...")
print("Last 20 elements of the first training review after padding:\n", train_data[0][-20:])


Length of the first training review after padding: 256
Length of the second training review after padding: 256

First training review after padding (first 20 elements):
 [   1   14   22   16   43  530  973 1622 1385   65  458 4468   66 3941
    4  173   36  256    5   25] ...
Last 20 elements of the first training review after padding:
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


##simple rnn


In [ ]:
from tensorflow.keras.optimizers import Adam,SGD, Adagrad, RMSprop
from tensorflow.keras.layers import Dense, Activation, Dropout,Embedding,SimpleRNN,LSTM,GRU
from sklearn.linear_model import LogisticRegression
from tensorflow.keras.models import Sequential
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
import time
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
import numpy as np
from tensorflow.keras.layers import Input, Embedding, GRU, Dense # Added Input to imports
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay


In [ ]:

rnn_model=Sequential([
    Embedding(input_dim=10000, output_dim=128, input_length=maxlen),#input size equal size of longethest message that is max_sequence_length
    SimpleRNN(128),
    Dense(1,activation="sigmoid"),#output layer

])

rnn_model.compile(loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)
start_time = time.time()
history = rnn_model.fit(
    train_data, train_labels,
    epochs=5,
    batch_size=64,
    validation_split=0.2 # 80% of training data → training and 20% of training data → validation
)
end_time = time.time()

rnn_training_time = end_time - start_time

print("Total Training Time:", rnn_training_time, "seconds")

Epoch 1/5


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


313/313 ━━━━━━━━━━━━━━━━━━━━ 68s 209ms/step - accuracy: 0.5108 - loss: 0.6968 - val_accuracy: 0.5298 - val_loss: 0.6888
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 69s 221ms/step - accuracy: 0.5485 - loss: 0.6733 - val_accuracy: 0.5328 - val_loss: 0.6748
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 79s 252ms/step - accuracy: 0.5888 - loss: 0.6346 - val_accuracy: 0.5542 - val_loss: 0.6825
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 72s 221ms/step - accuracy: 0.6112 - loss: 0.5867 - val_accuracy: 0.5466 - val_loss: 0.6920
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 91s 249ms/step - accuracy: 0.6371 - loss: 0.5586 - val_accuracy: 0.5386 - val_loss: 0.7282
Total Training Time: 378.43497824668884 seconds


In [ ]:
rnn_test_loss, rnn_test_acc = rnn_model.evaluate(test_data, test_labels)

print("Test accuracy:",rnn_test_acc)

782/782 ━━━━━━━━━━━━━━━━━━━━ 38s 48ms/step - accuracy: 0.5205 - loss: 0.7380
Test accuracy: 0.5204799771308899


### LSTM

In [ ]:


LSTM_model = Sequential([
    Embedding(input_dim=10000, output_dim=128, input_length=maxlen),
    LSTM(128),
    Dense(1, activation='sigmoid')
])
LSTM_model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)
start_time = time.time()
history = LSTM_model.fit(
    train_data, train_labels,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)
end_time = time.time()

lstm_training_time = end_time - start_time

print("Total Training Time:", lstm_training_time, "seconds")

Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 222s 702ms/step - accuracy: 0.5271 - loss: 0.6913 - val_accuracy: 0.5650 - val_loss: 0.6570
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 233s 746ms/step - accuracy: 0.5973 - loss: 0.6187 - val_accuracy: 0.5640 - val_loss: 0.6544
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 242s 684ms/step - accuracy: 0.6278 - loss: 0.5876 - val_accuracy: 0.7802 - val_loss: 0.6618
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 214s 684ms/step - accuracy: 0.7574 - loss: 0.5344 - val_accuracy: 0.8054 - val_loss: 0.4767
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 218s 697ms/step - accuracy: 0.8671 - loss: 0.3409 - val_accuracy: 0.8478 - val_loss: 0.3817
Total Training Time: 1130.2265038490295 seconds


In [ ]:
lstm_test_loss, lstm_test_acc = LSTM_model.evaluate(test_data, test_labels)

print("Test accuracy:", lstm_test_acc)

782/782 ━━━━━━━━━━━━━━━━━━━━ 106s 136ms/step - accuracy: 0.8397 - loss: 0.4008
Test accuracy: 0.839680016040802


## GRU

In [ ]:
#initialize model

gru_model = Sequential(
    [Input(shape=(maxlen,)),
    Embedding(input_dim=10000, output_dim=128),
    GRU(128),
    Dense(1, activation='sigmoid')]
)
gru_model.compile(loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy'])
#training
start = time.time()
history = gru_model.fit(
    train_data,
    train_labels,
    epochs = 5,
    batch_size = 64,
    validation_split = 0.2
)
end = time.time()

gru_training_time = end - start
print("GRU Model:")
print(f"Training finished in {gru_training_time:.2f} sec")

Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 223s 705ms/step - accuracy: 0.5343 - loss: 0.6800 - val_accuracy: 0.5660 - val_loss: 0.6563
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 214s 683ms/step - accuracy: 0.6661 - loss: 0.5768 - val_accuracy: 0.7842 - val_loss: 0.4779
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 209s 668ms/step - accuracy: 0.8670 - loss: 0.3234 - val_accuracy: 0.8598 - val_loss: 0.3405
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 209s 668ms/step - accuracy: 0.9343 - loss: 0.1815 - val_accuracy: 0.8874 - val_loss: 0.2791
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 215s 687ms/step - accuracy: 0.9678 - loss: 0.1028 - val_accuracy: 0.8762 - val_loss: 0.3371
GRU Model:
Training finished in 1069.79 sec


In [ ]:
#testing
gru_loss,gru_accuracy = gru_model.evaluate(test_data,test_labels)
print(f"GRU test info: Test Accuracy = {gru_accuracy*100:.2f}% with Test Loss = {gru_loss:.2f}")

782/782 ━━━━━━━━━━━━━━━━━━━━ 57s 73ms/step - accuracy: 0.8689 - loss: 0.3642
GRU test info: Test Accuracy = 86.89% with Test Loss = 0.36


In [ ]:
# Compare Models

results = {}
for name, model, t in [
    ("Simple RNN", rnn_model,  rnn_training_time),
    ("LSTM",       LSTM_model, lstm_training_time),
    ("GRU",        gru_model,  gru_training_time),
]:
    loss, acc = model.evaluate(test_data, test_labels, verbose=0)
    results[name] = {"loss": loss, "accuracy": acc, "time": t}

    #  Confusion Matrix
    preds = (model.predict(test_data, verbose=0) > 0.5).astype(int).flatten()

    print(f"\n{'='*50}")
    print(f"  {name}")
    print('='*50)

    print("\nConfusion Matrix:")
    print(confusion_matrix(test_labels, preds))

    print("\nClassification Report:")
    print(classification_report(test_labels, preds, target_names=["Negative", "Positive"]))


  Simple RNN

Confusion Matrix:
[[10921  1579]
 [10409  2091]]

Classification Report:
              precision    recall  f1-score   support

    Negative       0.51      0.87      0.65     12500
    Positive       0.57      0.17      0.26     12500

    accuracy                           0.52     25000
   macro avg       0.54      0.52      0.45     25000
weighted avg       0.54      0.52      0.45     25000


  LSTM

Confusion Matrix:
[[11139  1361]
 [ 2647  9853]]

Classification Report:
              precision    recall  f1-score   support

    Negative       0.81      0.89      0.85     12500
    Positive       0.88      0.79      0.83     12500

    accuracy                           0.84     25000
   macro avg       0.84      0.84      0.84     25000
weighted avg       0.84      0.84      0.84     25000


  GRU

Confusion Matrix:
[[10971  1529]
 [ 1748 10752]]

Classification Report:
              precision    recall  f1-score   support

    Negative       0.86      0.88      0

In [ ]:
# which model performed best and why
best_name = max(results, key=lambda n: results[n]['accuracy'])
print(f"Best Model: {best_name}  |  Accuracy: {results[best_name]['accuracy']:.4f}")

print("""
Simple RNN  → Fastest to train but weakest — suffers from Vanishing Gradient,
              meaning it forgets context from earlier in the sequence.

LSTM        → Best at capturing long-range dependencies using 3 gates
              (forget / input / output), but slowest to train.

GRU         → Accuracy close to LSTM using only 2 gates (reset / update),
              faster and lighter than LSTM.

GRU and LSTM both outperform Simple RNN because movie reviews require
understanding context spread across many words — for example, the word
"not" at the beginning of a sentence changes its entire meaning.
The Simple RNN forgets this due to the Vanishing Gradient problem.

Conclusion: GRU is the best choice — high accuracy with faster training than LSTM.
""")

Best Model: GRU  |  Accuracy: 0.8689

Simple RNN  → Fastest to train but weakest — suffers from Vanishing Gradient,
              meaning it forgets context from earlier in the sequence.

LSTM        → Best at capturing long-range dependencies using 3 gates
              (forget / input / output), but slowest to train.

GRU         → Accuracy close to LSTM using only 2 gates (reset / update),
              faster and lighter than LSTM.

GRU and LSTM both outperform Simple RNN because movie reviews require
understanding context spread across many words — for example, the word
"not" at the beginning of a sentence changes its entire meaning.
The Simple RNN forgets this due to the Vanishing Gradient problem.

Conclusion: GRU is the best choice — high accuracy with faster training than LSTM.



In [ ]:
# Correct vs Wrong Predictions
best_model = {"Simple RNN": rnn_model, "LSTM": LSTM_model, "GRU": gru_model}[best_name]

probs = best_model.predict(test_data, verbose=0).flatten()
preds = (probs > 0.5).astype(int)

correct_idx   = np.where(preds == test_labels)[0]
incorrect_idx = np.where(preds != test_labels)[0]

label_map = {0: "NEGATIVE", 1: "POSITIVE"}

def show_examples(indices, title, n=3):
    print(f"\n{'='*60}")
    print(f"  {title}  ({len(indices)} total)")
    print('='*60)
    for i, idx in enumerate(indices[:n]):
        snippet = ' '.join(decode_review(test_data[idx]).split()[:40])
        print(f"\nExample {i+1}:")
        print(f"  Review    : {snippet} ...")
        print(f"  True      : {label_map[test_labels[idx]]}")
        print(f"  Predicted : {label_map[preds[idx]]}  (confidence: {probs[idx]:.3f})")

show_examples(correct_idx,   " CORRECT PREDICTIONS")
show_examples(incorrect_idx, " WRONG PREDICTIONS")


   CORRECT PREDICTIONS  (21723 total)

Example 1:
  Review    : <START> please give this one a miss br br <UNK> <UNK> and the rest of the cast rendered terrible performances the show is flat flat flat br br i don't know how michael madison could have allowed this one on ...
  True      : NEGATIVE
  Predicted : NEGATIVE  (confidence: 0.417)

Example 2:
  Review    : a lot of patience because it focuses on mood and character development the plot is very simple and many of the scenes take place on the same set in frances <UNK> the sandy dennis character apartment but the film builds ...
  True      : POSITIVE
  Predicted : POSITIVE  (confidence: 0.993)

Example 3:
  Review    : <START> like some other people wrote i'm a die hard mario fan and i loved this game br br this game starts slightly boring but trust me it's worth it as soon as you start your hooked the levels are ...
  True      : POSITIVE
  Predicted : POSITIVE  (confidence: 0.997)

   WRONG PREDICTIONS  (3277 total)

Example 1

In [ ]:
#  Prediction Demo
def predict_sentiment(review_text, model=best_model):
    tokens = [word_index.get(w.lower(), word_index["<UNK>"]) for w in review_text.split()]
    tokens = [t if t < 10000 else word_index["<UNK>"] for t in tokens]
    padded = pad_sequences([tokens], value=word_index["<PAD>"], padding='post', maxlen=maxlen)
    prob   = model.predict(padded, verbose=0)[0][0]
    label  = "POSITIVE " if prob > 0.5 else "NEGATIVE "
    print(f"\nReview    : {review_text}")
    print(f"Sentiment : {label}")
    print(f"Confidence: {prob:.4f}")

predict_sentiment("This movie was absolutely fantastic! The acting was superb.")
predict_sentiment("What a terrible film. The plot made no sense at all.")

user_review = input("Enter your movie review: ")
predict_sentiment(user_review)


Review    : This movie was absolutely fantastic! The acting was superb.
Sentiment : NEGATIVE 
Confidence: 0.1830

Review    : What a terrible film. The plot made no sense at all.
Sentiment : NEGATIVE 
Confidence: 0.0701
Enter your movie review: I wanted to hate this film but somehow I ended up enjoying it.

Review    : I wanted to hate this film but somehow I ended up enjoying it.
Sentiment : POSITIVE 
Confidence: 0.5689
